# ToneFit — FaRL vs SVM Comparison (Kaggle)

**Research question:** Can a traditional CV baseline (SVM on handcrafted color features) match or approach a face-specialized deep learning model (FaRL) on personal color season classification?

Dataset: Deep Armocromia (Stacchio et al., ECCV 2024)

---

**Before running:**
1. Set accelerator to **GPU T4** (Settings → Accelerator → GPU T4)
2. Add the RGB-M dataset (Settings → Add Data → your uploaded RGB-M dataset)
3. Make sure your latest code is pushed to GitHub before cloning

**Pipeline:**
1. Check GPU
2. Clone repo + install dependencies
3. Link RGB-M dataset from Kaggle input
4. Preprocessing — extract CIELab/HSV features
5. Train FaRL (Deep Learning baseline)
6. Train SVM (Classical CV baseline)
7. Comparison stats — accuracy, F1, confusion matrices, Autumn analysis
8. Save results

---
## Step 0 — Check GPU

In [ ]:
import torch

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU detected — go to Settings -> Accelerator -> GPU T4')

---
## Step 1 — Clone Repo & Install Dependencies

In [ ]:
import os

REPO_URL = 'https://github.com/ajipal/ToneFit.git'
REPO_DIR = '/kaggle/working/ToneFit'

if os.path.isdir(REPO_DIR):
    print('Repo already cloned. Pulling latest...')
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')
!ls

In [ ]:
!pip install -q timm>=0.9.0 scikit-image imagehash seaborn joblib
!pip install -q git+https://github.com/openai/CLIP.git
print('Dependencies installed')

---
## Step 2 — Link RGB-M Dataset from Kaggle Input

> Update `KAGGLE_INPUT_PATH` to match the path of your uploaded RGB-M dataset.

In [ ]:
import os

KAGGLE_INPUT_PATH = '/kaggle/input/datasets/ronliwag/tonefit-rgb-m/Projects/ToneFit/RGB-M'
LOCAL_LINK        = '/kaggle/working/ToneFit/RGB-M'

if os.path.lexists(LOCAL_LINK):
    os.remove(LOCAL_LINK)

os.symlink(KAGGLE_INPUT_PATH, LOCAL_LINK)
print(f'Linked: {LOCAL_LINK} -> {KAGGLE_INPUT_PATH}')

# Verify structure
for split in ['train', 'test']:
    split_path = os.path.join(LOCAL_LINK, split)
    if os.path.exists(split_path):
        total = sum(
            len(files)
            for _, _, files in os.walk(split_path)
        )
        print(f'  {split}/: {total} images')
    else:
        print(f'  {split}/ NOT FOUND — check KAGGLE_INPUT_PATH')

---
## Step 3 — Preprocessing

> Extracts CIELab/HSV color features from skin regions for both FaRL (class weights)
> and SVM (feature input). Required before SVM training.

In [ ]:
import sys, importlib

if 'preprocess' in sys.modules:
    importlib.reload(sys.modules['preprocess'])
else:
    import preprocess

preprocess.run()
print('Preprocessing complete')

In [ ]:
import pandas as pd

df = pd.read_csv('features.csv')
print(f'features.csv: {len(df)} rows')
print('\nClass distribution:')
print(df.groupby(['split', 'season']).size().unstack())

---
## Step 4 — Train FaRL (Deep Learning Baseline)

> FaRL pretrained weights (~650 MB) are downloaded from GitHub releases.  
> Falls back to ResNeXt50 automatically if download fails.  
> Expected training time: ~30–45 minutes on T4 GPU.

In [ ]:
import os
os.makedirs('models', exist_ok=True)

if not os.path.exists('models/farl_weights.pth'):
    !wget -q --show-progress -O models/farl_weights.pth \
      "https://github.com/FacePerceiver/FaRL/releases/download/pretrained_weights/FaRL-Base-Patch16-LAIONFace20M-ep64.pth"
    print(f'Downloaded: {os.path.getsize("models/farl_weights.pth")/1e6:.0f} MB')
else:
    print(f'FaRL weights already present: {os.path.getsize("models/farl_weights.pth")/1e6:.0f} MB')

In [ ]:
import sys, importlib

if 'train_farl' in sys.modules:
    importlib.reload(sys.modules['train_farl'])
else:
    import train_farl

train_farl.main()
print('FaRL training complete')

In [ ]:
from IPython.display import Image as IPImage, display
import os

if os.path.exists('results/farl_training.png'):
    display(IPImage('results/farl_training.png'))
else:
    print('Training curves not found — check that training completed successfully')

---
## Step 5 — Train SVM (Classical CV Baseline)

> SVM with RBF kernel trained on 7 handcrafted color features: L*, a*, b*, ITA, H, S, V.  
> GridSearchCV (5-fold) over C and gamma. No GPU required.  
> Expected training time: 1–3 minutes on CPU.

In [ ]:
import sys, importlib

if 'train_svm' in sys.modules:
    importlib.reload(sys.modules['train_svm'])
else:
    import train_svm

train_svm.main()
print('SVM training complete')

---
## Step 6 — Comparison Stats

> Reloads both trained models, evaluates on the test set, and produces:
> - Comparison table vs paper baselines (Stacchio et al., 2024)
> - Confusion matrices side by side
> - Bar chart (Accuracy + F1)
> - Per-class Autumn recall analysis

In [ ]:
import os, json, pickle
import numpy as np
import pandas as pd
import torch
import joblib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from torch.utils.data import TensorDataset, DataLoader as CachedDL
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
)

SEASONS    = ['autumn', 'spring', 'summer', 'winter']
COLOR_COLS = ['L_mean', 'a_mean', 'b_mean', 'ITA', 'H_mean', 'S_mean', 'V_mean']
_ALL_COLS  = ['L_mean','a_mean','b_mean','L_std','a_std','b_std',
              'ITA','H_mean','S_mean','V_mean','skin_ratio','blur_score']

os.makedirs('results', exist_ok=True)
print('Libraries loaded')


In [ ]:
# --- Reload FaRL model and evaluate on test set ---
import importlib, train_farl
importlib.reload(train_farl)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

farl_model, backbone_name = train_farl.load_model()
farl_model = farl_model.to(device)

ckpt = torch.load('models/farl_model.pth', map_location=device)
farl_model.shared.load_state_dict(ckpt['shared'])
farl_model.season_head.load_state_dict(ckpt['season_head'])
farl_model.subtype_head.load_state_dict(ckpt['subtype_head'])

# Load pre-cached test features (written by train_farl.main())
te = torch.load(train_farl.CACHE_TEST, map_location=device)
te_ds = TensorDataset(te['features'].to(device), te['season_labels'].to(device), te['subtype_labels'].to(device))
test_loader_cached = CachedDL(te_ds, batch_size=64, shuffle=False)

# Returns (szn_true, szn_pred, sub_true, sub_pred)
szn_true, szn_pred, sub_true, sub_pred = train_farl.get_all_predictions_cached(farl_model, test_loader_cached)
y_true_farl, y_pred_farl = szn_true, szn_pred

farl_acc  = accuracy_score(y_true_farl, y_pred_farl)
farl_prec = precision_score(y_true_farl, y_pred_farl, average='weighted', zero_division=0)
farl_rec  = recall_score(y_true_farl, y_pred_farl, average='weighted', zero_division=0)
farl_f1   = f1_score(y_true_farl, y_pred_farl, average='weighted', zero_division=0)
farl_cm   = confusion_matrix(y_true_farl, y_pred_farl)
farl_autumn_recall = farl_cm[0, 0] / farl_cm[0].sum()

with open('results/farl_history.json') as f:
    farl_hist = json.load(f)

print(f'FaRL Backbone    : {backbone_name}')
print(f'FaRL Season Acc  : {farl_acc:.4f}')
print(f'FaRL F1          : {farl_f1:.4f}')
print(f'FaRL Autumn Rec  : {farl_autumn_recall:.4f}')
print(f'FaRL SubType Acc : {farl_hist.get("best_val_subtype_acc", "N/A")}')


In [ ]:
# --- Reload SVM model and evaluate on test set ---
df = pd.read_csv('features.csv')
with open('models/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

col_idx = [_ALL_COLS.index(c) for c in COLOR_COLS]
test_df = df[df['split'] == 'test']
scaled  = scaler.transform(test_df[_ALL_COLS].values.astype(np.float32))
X_test  = scaled[:, col_idx]
y_test  = test_df['label'].values

svm_model = joblib.load('models/svm_model.pkl')
y_pred_svm  = svm_model.predict(X_test)
y_proba_svm = svm_model.predict_proba(X_test)

svm_acc  = accuracy_score(y_test, y_pred_svm)
svm_prec = precision_score(y_test, y_pred_svm, average='weighted', zero_division=0)
svm_rec  = recall_score(y_test, y_pred_svm, average='weighted', zero_division=0)
svm_f1   = f1_score(y_test, y_pred_svm, average='weighted', zero_division=0)
svm_cm   = confusion_matrix(y_test, y_pred_svm)
svm_autumn_recall = svm_cm[0, 0] / svm_cm[0].sum()
svm_top2 = np.mean([y_test[i] in np.argsort(y_proba_svm[i])[-2:] for i in range(len(y_test))])

# Load saved SVM results for best params
with open('results/svm_results.json') as f:
    svm_res = json.load(f)

print(f'SVM Best params  : {svm_res["best_params"]}')
print(f'SVM Accuracy     : {svm_acc:.4f}')
print(f'SVM F1           : {svm_f1:.4f}')
print(f'SVM Autumn recall: {svm_autumn_recall:.4f}')

In [ ]:
# --- Comparison table including paper baselines ---
PAPER_BASELINES = [
    {'Model': 'FaRL16',    'Source': 'Stacchio et al. (2024)', 'Accuracy': 0.525, 'F1': 0.516, 'Top-2': 0.815, 'Autumn Recall': '-'},
    {'Model': 'FaRL64',    'Source': 'Stacchio et al. (2024)', 'Accuracy': 0.554, 'F1': 0.548, 'Top-2': 0.808, 'Autumn Recall': '-'},
    {'Model': 'ResNeXt50', 'Source': 'Stacchio et al. (2024)', 'Accuracy': 0.513, 'F1': 0.502, 'Top-2': 0.789, 'Autumn Recall': '-'},
]

# FaRL top-2 accuracy - run season head on cached features
farl_proba_all = []
farl_model.season_head.eval()
with torch.no_grad():
    for features, _, __ in test_loader_cached:
        shared = farl_model.shared(features)
        logits = farl_model.season_head(shared)
        farl_proba_all.append(torch.softmax(logits, dim=1).cpu().numpy())
farl_proba_all = np.concatenate(farl_proba_all)
farl_top2 = float(np.mean([
    y_true_farl[i] in np.argsort(farl_proba_all[i])[-2:]
    for i in range(len(y_true_farl))
]))

OUR_RESULTS = [
    {
        'Model': f'FaRL (Ours - {backbone_name.split("(")[0].strip()})',
        'Source': 'This study',
        'Accuracy': round(farl_acc, 4),
        'F1': round(farl_f1, 4),
        'Top-2': round(farl_top2, 4),
        'Autumn Recall': round(farl_autumn_recall, 4),
    },
    {
        'Model': 'SVM RBF - Color Features (Ours)',
        'Source': 'This study',
        'Accuracy': round(svm_acc, 4),
        'F1': round(svm_f1, 4),
        'Top-2': round(svm_top2, 4),
        'Autumn Recall': round(svm_autumn_recall, 4),
    },
]

comparison_df = pd.DataFrame(PAPER_BASELINES + OUR_RESULTS)
comparison_df.to_csv('results/comparison_farl_svm.csv', index=False)

print('\n' + '='*75)
print('  MODEL COMPARISON - FaRL vs SVM vs Paper Baselines')
print('='*75)
print(comparison_df.to_string(index=False))
print('='*75)
print('Saved: results/comparison_farl_svm.csv')


In [ ]:
# --- Confusion matrices side by side ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Confusion Matrices — FaRL vs SVM', fontsize=14, fontweight='bold')

season_labels = [s.capitalize() for s in SEASONS]

for ax, cm, title in [
    (axes[0], farl_cm, f'FaRL\n(Acc={farl_acc:.3f}, F1={farl_f1:.3f})'),
    (axes[1], svm_cm,  f'SVM — Color Features\n(Acc={svm_acc:.3f}, F1={svm_f1:.3f})'),
]:
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=season_labels, yticklabels=season_labels,
        ax=ax, annot_kws={'size': 12},
    )
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted Label')

plt.tight_layout()
plt.savefig('results/confusion_farl_vs_svm.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/confusion_farl_vs_svm.png')

In [ ]:
# --- Bar chart: all models including paper baselines ---
models = [
    'FaRL16\n(Paper)', 'FaRL64\n(Paper)', 'ResNeXt50\n(Paper)',
    'FaRL\n(Ours)', 'SVM\n(Ours)',
]
accuracies = [0.525, 0.554, 0.513, farl_acc, svm_acc]
f1_scores  = [0.516, 0.548, 0.502, farl_f1,  svm_f1]
colors     = ['#aec6cf', '#aec6cf', '#aec6cf', '#2196F3', '#FF5722']

x = np.arange(len(models))
w = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - w/2, accuracies, w, label='Accuracy', color=colors, alpha=0.9)
bars2 = ax.bar(x + w/2, f1_scores,  w, label='F1 Score',  color=colors, alpha=0.6)

# Value labels
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

# Paper best line
ax.axhline(y=0.554, color='gray', linestyle='--', linewidth=1.2,
           label='Paper best (FaRL64 Acc=0.554)')

ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=10)
ax.set_ylim(0, 0.72)
ax.set_ylabel('Score')
ax.set_title('FaRL vs SVM vs Paper Baselines', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Annotation: our models
ax.annotate('Our Models', xy=(3.5, 0.67), fontsize=10, color='#333',
            ha='center', style='italic')
ax.axvspan(2.5, 4.5, alpha=0.05, color='blue')

plt.tight_layout()
plt.savefig('results/barchart_farl_vs_svm.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/barchart_farl_vs_svm.png')

In [ ]:
# --- Per-class analysis — Autumn focus ---
from sklearn.metrics import classification_report

print('='*60)
print('  PER-CLASS ANALYSIS')
print('='*60)

print('\nFaRL:')
print(classification_report(y_true_farl, y_pred_farl, target_names=SEASONS, digits=4))

print('SVM — Color Features:')
print(classification_report(y_test, y_pred_svm, target_names=SEASONS, digits=4))

# Autumn confusion detail
print('='*60)
print('  AUTUMN MISCLASSIFICATION BREAKDOWN')
print('  (Paper reported 80 Autumn samples misclassified as Winter)')
print('='*60)

for model_name, cm, y_pred in [('FaRL', farl_cm, y_pred_farl), ('SVM', svm_cm, y_pred_svm)]:
    autumn_row = cm[0]
    total_autumn = autumn_row.sum()
    print(f'\n{model_name} — Autumn row (total={total_autumn}):')
    for j, s in enumerate(SEASONS):
        pct = autumn_row[j] / total_autumn * 100
        marker = ' <-- CORRECT' if j == 0 else (' <-- misclassified as Winter' if j == 3 else '')
        print(f'  Predicted {s:<8}: {autumn_row[j]:3d} ({pct:.1f}%){marker}')

# Summary
print('\n' + '='*60)
print('  AUTUMN RECALL SUMMARY')
print(f'  Paper (FaRL64)  : see confusion matrix Fig. 5')
print(f'  FaRL (Ours)     : {farl_autumn_recall:.4f}')
print(f'  SVM (Ours)      : {svm_autumn_recall:.4f}')
print('='*60)

In [ ]:
# --- Warm/Cool confusion analysis ---
# Warm seasons: Autumn (0), Spring (1)
# Cool seasons: Summer (2), Winter (3)
# Cross-temperature error = predicting warm as cool or vice versa

WARM = {0, 1}   # autumn, spring
COOL = {2, 3}   # summer, winter

def cross_temp_error_rate(y_true, y_pred):
    errors = sum(
        1 for yt, yp in zip(y_true, y_pred)
        if yt != yp and (
            (yt in WARM and yp in COOL) or
            (yt in COOL and yp in WARM)
        )
    )
    return errors / len(y_true)

farl_cross = cross_temp_error_rate(y_true_farl, y_pred_farl)
svm_cross  = cross_temp_error_rate(y_test, y_pred_svm)

print('='*60)
print('  WARM/COOL CROSS-TEMPERATURE ERROR RATE')
print('  (Key failure mode identified by Stacchio et al. 2024)')
print('='*60)
print(f'  FaRL (Ours)  : {farl_cross:.4f} ({farl_cross*100:.1f}% of all samples)')
print(f'  SVM  (Ours)  : {svm_cross:.4f}  ({svm_cross*100:.1f}% of all samples)')
print()
if svm_cross < farl_cross:
    print('  SVM has FEWER cross-temperature errors — handcrafted a* feature helps')
elif farl_cross < svm_cross:
    print('  FaRL has FEWER cross-temperature errors — deep features capture undertone better')
else:
    print('  Both models have equal cross-temperature error rates')

---
## Step 7 — Save Results

In [ ]:
import zipfile, glob

zip_path = '/kaggle/working/ToneFit_FaRL_SVM_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob('results/*'):
        z.write(f)
    for f in glob.glob('models/*.pth'):
        if 'farl_weights' not in f:
            z.write(f)
    if os.path.exists('models/svm_model.pkl'):
        z.write('models/svm_model.pkl')
    if os.path.exists('features.csv'):
        z.write('features.csv')

size_mb = os.path.getsize(zip_path) / (1024 * 1024)
print(f'Saved: {zip_path} ({size_mb:.1f} MB)')
print('Download from: Kaggle notebook -> Output panel -> ToneFit_FaRL_SVM_results.zip')